# ANLP — Session 2: LED fine-tuning

**What this run does**
- Fine-tunes `allenai/led-base-16384` end-to-end at **8192-token** training windows.
- Each epoch the dataloader samples a **different random 8k slice** of every match, so over 15 epochs the model sees the whole transcript piecewise. Validation always uses the first 8k tokens (deterministic, so early stopping is reliable).
- Inference runs at the **full 16384** context.
- Final eval covers ALL prediction JSONs present in `outputs/predictions/` (i.e. just `finetuned_led.json` from this session — comparison against run1/run2 is done locally after download).

**Estimated time:** ~3–5 h on T4. Train ~2.5 h, inference ~10–20 min, eval <1 min. Well inside Kaggle's 9h limit.

**Before running:**
- Sidebar → Accelerator → **GPU T4 x2** (we only use one card; the second is harmless. P100 sm_60 does NOT work with current PyTorch — must be T4.)
- Sidebar → Internet → **On**
- Save Version → **Save & Run All (Commit)** so outputs are preserved.

In [ ]:
import os
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
WORK = '/kaggle/working'
REPO = f'{WORK}/ANLP_Project'
if not os.path.isdir(REPO):
    !git clone --quiet --branch setup/local-run https://github.com/christiandalfarra/ANLP_Project.git $REPO
%cd $REPO
!git pull --quiet

In [ ]:
!pip install -q sentencepiece bert_score rouge_score 'transformers>=4.40' 'accelerate>=0.30'

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Wipe any old LED checkpoint & prediction file before re-training

In [ ]:
!rm -rf checkpoints/led
!rm -f outputs/predictions/finetuned_led.json

## Fine-tune LED (~2.5 h)

Random-window sampling: per epoch, each match's input is a randomly-positioned 8192-token slice of the full tokenised transcript. Val uses a fixed first-8k window so ROUGE is deterministic. Gradient checkpointing + fp16 keeps it inside T4's 15 GB.

In [ ]:
!python scripts/run_finetuning.py --model led --output_dir checkpoints/led

## LED inference on 9 test matches (~10–20 min)

Inference uses the full 16384-token context with beam=2 (beam=4 risks OOM at 16k on T4).

In [ ]:
!python scripts/run_inference_finetuned.py --model led --checkpoint checkpoints/led

## Evaluate (ROUGE-only — bert_score is buggy on Kaggle's transformers version)

In [ ]:
!python scripts/run_evaluation.py --no-bertscore

In [ ]:
import os, json, glob
for p in sorted(glob.glob('outputs/predictions/*.json')):
    print('===', os.path.basename(p))
    d = json.load(open(p))
    mid, summary = next(iter(d.items()))
    print(f'[{mid}]\n{summary[:600]}\n')
for p in sorted(glob.glob('outputs/results/*')):
    print(p)
    if p.endswith('.csv'):
        print(open(p).read())